# 01 - A tour of the Multi-Agent Particle Environments (MPE)

MPE is a set of small 2-D worlds: agents are circles that move by applying forces, see each other and some landmarks, and (in some tasks) talk. Tiny observations, tiny networks, fast to train - ideal for studying *multi-agent* effects without the compute of a big simulator.

Agents here act **randomly**. In the next notebooks we train them and watch what goes right and wrong.

## Three kinds of multi-agent problems

| Kind | Reward | Example below |
|---|---|---|
| **Cooperative** | everyone shares one goal | Coop Navigation, Speaker-Listener |
| **Competitive** | one side's gain is the other's loss | Predator *vs* prey |
| **Mixed** | teammates cooperate *and* compete with other teams | Predator-Prey (predators work together against the prey) |

All three tasks use **continuous actions**: each mover outputs 5 numbers in [0, 1] = force for *[no-op, left, right, down, up]*.

In [ ]:
%matplotlib inline
from marl.render import make_mpe, describe, rollout, save_video, show_video, play_live

## 1. Coop Navigation (`simple_spread`) - cooperative

*N* agents (blue) must **cover N landmarks** (black) between them, without bumping into each other.

* **Reward**: shared part = - sum over landmarks of the distance to the *closest* agent; local part = -1 per collision. `local_ratio` (default 0.5) mixes the two.
* **Observation**: own velocity and position, relative positions of all landmarks and of the other agents.
* **What makes it hard**: nobody tells an agent *which* landmark is "its" landmark - the team has to sort that out.

In [ ]:
env = make_mpe("spread")          # try make_mpe("spread", N=6)
describe(env)
frames, returns = rollout(env, seed=0, episodes=2)
show_video(frames)

## 2. Predator-Prey (`simple_tag`) - mixed

3 slow **predators** (red) chase 1 fast **prey** (green) around 2 obstacles (big black).

* **Predators**: +10 to *every* predator whenever any predator touches the prey (a team reward).
* **Prey**: -10 per touch, plus a penalty for running off the map. Prey is faster (max speed 1.3 vs 1.0, higher acceleration).
* **What makes it hard**: a single predator can't catch a faster prey - the predators have to coordinate. And the prey learns too, so what worked yesterday may stop working today.

In [ ]:
env = make_mpe("tag")
describe(env)
frames, returns = rollout(env, seed=0, episodes=2)
print({a: round(r, 1) for a, r in returns[0].items()})
show_video(frames)

## 3. Speaker-Listener (`simple_speaker_listener`) - cooperative, with communication

* The **speaker** (grey, can't move) sees the **colour of the goal landmark** and broadcasts a 3-number message.
* The **listener** (can move) sees the landmarks and the message, but **not** which one is the goal.
* **Reward** (shared): - squared distance from listener to the goal landmark.
* **What makes it hard**: the two have to *invent a language* while learning - and the listener depends entirely on what the speaker says. The text at the bottom of the video shows the message.

In [ ]:
env = make_mpe("speaker_listener")
describe(env)
frames, returns = rollout(env, seed=0, episodes=2)
show_video(frames)

## What we will look for when we train

* **Speaker-Listener** - both agents learn at the same time. What happens to the listener when the speaker keeps changing what its messages mean?
* **Predator-Prey** - predators get better at catching... until the prey gets better at escaping. Is a later predator always a better predator?
* **Coop Navigation** - do agents settle on who covers which landmark? What happens when we go from 3 to 6 agents - one network for everybody or one each?